# rl19_bridge

> Phase 19 (`specs/multiagent/phase19/`, T5a and T5b): A1's bridged world. `BRIDGE19` (one plain bridge per pair of squads, every fourth subtask coupled: `bridge1`, as Gates C1b, C1c and C1d settled P19-8) and `GATE_C1` (Gate C1's record); `bridgeWorld19` and `bridgeCells19` bridge a world or cells in two squads or more (a single squad stays unbridged, P19-9); `BridgeContestLog`, contest credit on a bridge board: a relayed liar's value is the liar's (P20-12), and a true holder's mirror post against a mirror lie is a border contest, paid through the receiving squad's later correct solves (P23-8, kept by the user, 2026-11-12); `contestColumns19`, the relay-aware evaluation columns with the border contests and P19-9's waiting counts; `bridgeStepStats`, a training step's `longestBridge` and relays per bridge turn.

In [ ]:
#| default_exp rl19_bridge

In [ ]:
#| hide
from nbdev.showdoc import *

## The world and Gate C1

Gate C1 passed on 2026-11-11 and the inert-bridge rule did not fire, so A1 trains in squads joined by bridges (P19-1). Gates C1b, C1c and C1d tried a bridge with evidence, a dedicated bridge and a window; none beat one plain bridge, so every arm gets `bridge1` (P19-8). A cell is bridged only when it has two squads or more: C1's `checkBridge` refuses a single squad (the held-out 8-agent cells in squads of 8, and `seq8`).

In [ ]:
#| export
import json

from explRL.contest import Contest, ContestLog
from explRL.contest_eval import NAN, answersOf, waitAnswers

BRIDGE19 = (('couple', 0.25), ('bridges', 1))    # bridge1 on every arm (P19-8, settled by Gates C1b, C1c and C1d)
GATE_C1 = dict(passed=True, inertFired=False, userAcceptsBridges=None, bridges='bridge1', recorded='2026-11-12',
               result='bridge1 - colonyGuards +0.105 [0.086, 0.125], p 0.004; margin +0.020, p 0.047; bridge1 - coupled +0.013 '
                      '(the inert-bridge rule did not fire); bridge configuration settled by Gates C1b, C1c and C1d (P19-8)')


def bridgeWorld19(world, N):
    """`world` (a dict, or RL19Config's ((key, value), ...)) with BRIDGE19's keys when it is in at least two squads (N // squads
    >= 2); a single squad stays unbridged (P19-9: checkBridge refuses it). Returned in the form given."""
    w, S = dict(world), dict(world).get('squads')
    if S and int(N) // int(S) >= 2:
        w.update(BRIDGE19)
    return tuple(w.items()) if isinstance(world, tuple) else w


def bridgeCells19(cells, on=True):
    "Cells (clueCell dicts), each world bridged by bridgeWorld19 in place where `on` (a bool, or cell -> bool); the cells returned."
    for c in cells:
        if on(c) if callable(on) else on:
            c['cfg']['world'] = bridgeWorld19(c['cfg']['world'], c['N'])
    return cells


def isBridge19(world):
    "A Phase 19 cell (scheme 'contest', enforceK or p19) on a bridged board (couple): its contest columns are relay-aware."
    return 'couple' in world and (world.get('scheme') == 'contest' or 'enforceK' in world or bool(world.get('p19')))

## Contest credit on a bridge board

Two changes to `contest.ContestLog`, built only on a bridge board with contest credit (`BridgeBoardEnv.reset`):

- **Relayed lies (P20-12, FR-A9).** A relay is posted under the bridge's slot, but the value is its source's. `lies` reads each value's first poster through `relayLog`, looked up when a contest is judged (the relay's own `onShare` comes before `relayLog` is written). A relayed liar's value is a lie on the receiving squad's board, and a true post against it is a contest.
- **Border contests (P23-8, kept by the user on 2026-11-12).** A true holder's mirror post while the mirror shows a false value that a liar posted there first. Its watchers are the receiving squad's honest agents without the clue that were shown a lie of the slot before any true value: on their own board after a relay, or on the mirror for the pair's bridge (the user chose this over bridges only and a flat payment, 2026-11-12). Each watcher's later correct solve pays the contester as any contest does: 0.5, halved every `halfLife` turns of the lie's lead.

In [ ]:
#| export
def relayPosters(board):
    "{(t, bridge, k, j, r): the source post's poster} for every relay on the board (relayLog; {} on other boards)."
    return {tuple(x[:5]): x[5][1] for x in getattr(board, 'relayLog', ())}


class BridgeContestLog(ContestLog):
    """ContestLog on a bridge board (T5a; P20-12, and P23-8 as the user kept it). A relayed value is its source poster's, looked
    up when a contest is judged (relayLog is written after the relay's own onShare), so a relayed liar's value is a lie on the
    receiving squad's board and contesting it there pays, as any contest. A border contest: the first true mirror post, by an
    honest holder, while the mirror shows a false value a liar posted there first and no true value (one per slot, so repeated
    posts cannot farm it). Its watchers are the receiving squad's honest agents without the clue that were shown that mirror
    lie before any true value of the slot: on their own board through a relay, or, for the pair's bridge, on the mirror. A
    watcher's later correct solve pays the border contester as any contest pays (the user, 2026-11-12). Board watchers
    (`sawLie`) and border watchers (`sawBorder`) are kept apart."""
    def __init__(self, board, liarSlots, contestSlots, halfLife=4.0):
        super().__init__(board, liarSlots, contestSlots, halfLife)
        self.border, self.sawBorder, self.borderTrue = [], {}, set()   # border contests' indices; {(k, j): {i: t}}; {(i, k, j)}
        self.mirrorFirst, self._src = {}, (0, {})                       # {(k, j): {r: (poster, t)}}; relayPosters by relayLog length

    def _relays(self):
        n = len(self.board.relayLog)
        if self._src[0] != n:
            self._src = (n, relayPosters(self.board))
        return self._src[1]

    def lies(self, k, j):
        "[(t, r, liar)]: the slot's false values on the board first posted by a liar or relayed from a liar's mirror post."
        tr, src = self._true(k, j), self._relays()
        out = [(t, r, src.get((t, a, k, j, r), a)) for r, (a, t) in self.first.get((k, j), {}).items() if r != tr]
        return sorted(x for x in out if x[2] in self.liars)

    def mirrorLies(self, k, j):
        "[(t, r, liar)]: the false values first posted on the slot's mirror by a liar, oldest first."
        tr = self._true(k, j)
        return sorted((t, r, a) for r, (a, t) in self.mirrorFirst.get((k, j), {}).items() if a in self.liars and r != tr)

    def onShown(self, t, i, shown):
        """ContestLog.onShown on the board (relay-aware lies); then the border watchers: a mirror lie shown to i through a relay
        on its board, or on the mirror when i is the pair's bridge, before any true value of the slot."""
        super().onShown(t, i, shown)
        if i in self.liars:
            return
        b, src, p = self.board, self._relays(), self.board.bridgeOf(i)
        mir = {(k, j): set(v) for (k, j), v in self.mirrorFirst.items() if p is not None and k in b.open and b.coupled.get(k) == p}
        for (k, j) in set(mir) | {key for key in shown if key[0] in b.coupled}:
            if (i, k, j) in self.borderTrue:
                continue
            onBoard = shown.get((k, j), set())
            relayed = {r for r, (a, tt) in self.first.get((k, j), {}).items() if r in onBoard and (tt, a, k, j, r) in src}
            if self._true(k, j) in onBoard | mir.get((k, j), set()):
                self.borderTrue.add((i, k, j))
            elif i not in b.tasks[k].holders[j] and any(r in relayed | mir.get((k, j), set()) for _, r, _ in self.mirrorLies(k, j)):
                self.sawBorder.setdefault((k, j), {}).setdefault(i, float(t))

    def onMirror(self, t, i, k, j, r, true):
        "Agent i posts r on k's mirror at t, judged before the post counts: a border contest if it is the first true value against a mirror lie."
        lies, s, first = self.mirrorLies(k, j), self.board.coupled[k][0], self.mirrorFirst.setdefault((k, j), {})
        if true and lies and i not in self.liars and self._true(k, j) not in first:
            w = {a: None for a in self.sawBorder.get((k, j), {}) if a != i and a % self.board.nSquads == s and (a, k, j) not in self.borderTrue}
            self.border.append(len(self.contests))
            self.contests.append(Contest(float(t), i, k, j, lies[0][0], lies[0][1], lies[0][2], w))
        first.setdefault(int(r), (i, float(t)))

## Relay-aware evaluation columns

`contest_eval.contestColumns` hands A1's bridged cells (`isBridge19`: Phase 19 cells with `couple`) to `contestColumns19`; earlier bridged rows (C1 to C1d, with no Phase 19 marker) keep the plain columns, so their saved rows replay unchanged. The board is read through `Relayed`, where each relay is its source's post, so liar posts, contests, guarded lies, lags and uncorrected clues agree with the reward. Per-slot shares stay the agents' own. P19-9: coupled subtasks' answers leave the waiting rate's counts and are reported apart, and `enforceK` exempts them (`contest_eval.refuses`). `misreport19` does not count a relay as an honest false post; `rl19_measures` reads it where present.

In [ ]:
#| export
def attributed(board):
    "board.shareLog with each relay's poster replaced by its source post's poster (a relayed value is its source's)."
    src = relayPosters(board)
    return [s if tuple(s[:5]) not in src else (s[0], src[tuple(s[:5])]) + tuple(s[2:]) for s in board.shareLog]


class Relayed:
    "A board seen through `attributed` (or with the given `shareLog`) and other `submits` if given; every other attribute is the board's."
    def __init__(self, board, submits=None, shareLog=None):
        self._b, self.shareLog = board, attributed(board) if shareLog is None else shareLog
        self.submits = board.submits if submits is None else submits

    def __getattr__(self, name):
        return getattr(self._b, name)


def contestsOf19(board, nLiar):
    """contest_eval.contestsOf on a bridge board: a lie's poster read through relays (a relayed liar's value is the liar's),
    the contester the real poster (a bridge relaying the true value contests, as BridgeContestLog pays it)."""
    src, first, out = relayPosters(board), {}, []
    for s in board.shareLog:
        t, i, k, j, r, false = s[0], s[1], s[2], s[3], s[4], s[6]
        lies = sorted(tt for a, tt, f in first.get((k, j), {}).values() if a < nLiar and f)
        if not false and lies and i >= nLiar:
            out.append((float(t), i, k, j, lies[0]))
        first.setdefault((k, j), {}).setdefault(r, (src.get(tuple(s[:5]), i), float(t), false))
    return out


def borderContestsOf(board, nLiar):
    """[(t_c, contester, k, j, t_l)]: the border contests on a finished bridge board, as BridgeContestLog judges them (any
    scheme): the first true mirror post, by an honest slot, while the mirror shows a false value a liar slot posted there first."""
    first, out = {}, []
    for t, i, k, j, r, _, false in getattr(board, 'mirrorLog', ()):
        f = first.setdefault((k, j), {})
        lies = sorted(tt for a, tt, fl in f.values() if a < nLiar and fl)
        if not false and lies and i >= nLiar and not any(not fl for _, _, fl in f.values()):
            out.append((float(t), i, k, j, lies[0]))
        f.setdefault(r, (i, float(t), false))
    return out


def misreport19(board, nLiar):
    "Misreporting without relays: false values posted by honest agents per 100 honest shares, a relay not counted as a post."
    src = relayPosters(board)
    hs = [s for s in board.shareLog if s[1] >= nLiar and tuple(s[:5]) not in src]
    return 100.0 * sum(s[6] for s in hs) / len(hs) if hs else NAN


def contestColumns19(cfg, board, slotLogs=None):
    """contest_eval's columns on a bridged Phase 19 board, relay-aware (FR-A9). The lie side reads each relay as its source's post:
    liar posts, guarded lies and liar-first clues left uncorrected (`Relayed`); contests (`contestsOf19`) keep the real
    contester. Measures that order posts within a turn by slot (misinformed first answers, the waiting rate) read the board as
    it is. P19-9: the waiting rate's counts leave out coupled subtasks' answers, reported apart (waitAnswersCoupled,
    usedAnswersCoupled). Added: border contests (count, lags, per slot), mirror lies and misreport19."""
    from explRL.contest_eval import _columns
    nl, g, N, ans, cp = cfg.nLiar, int(cfg.world.get('guards', 0)), cfg.N, answersOf(slotLogs), getattr(board, 'coupled', {})
    out, rel = _columns(cfg, board, slotLogs), _columns(cfg, Relayed(board), slotLogs)
    out.update({k: rel[k] for k in ('liarPosts', 'guardLies', 'liarFirst', 'uncorrected')})
    gs, cs = set(range(nl, nl + g)), contestsOf19(board, nl)
    out.update(contests=len(cs), guardContests=sum(bool(gs & set(board.tasks[c[2]].holders[c[3]])) for c in cs),
               contestLags=json.dumps([c[0] - c[4] for c in cs]), agentContests=json.dumps([int(sum(1 for c in cs if c[1] == i)) for i in range(N)]))
    for suf, keep in (('', lambda k: k not in cp), ('Coupled', lambda k: k in cp)):
        w, u = waitAnswers(Relayed(board, [s for s in board.submits if keep(s[2])], board.shareLog), nl, ans, g)
        out['waitAnswers' + suf], out['usedAnswers' + suf] = w, u
    bc = borderContestsOf(board, nl)
    return dict(out, borderContests=len(bc), borderLags=json.dumps([c[0] - c[4] for c in bc]),
                agentBorderContests=json.dumps([int(sum(1 for c in bc if c[1] == i)) for i in range(N)]),
                mirrorLies=sum(1 for m in getattr(board, 'mirrorLog', ()) if m[1] < nl and m[6]), misreport19=misreport19(board, nl))

## A training step's bridge numbers

`trainRL17` logs `bridgeStepStats` in each step's metrics. The speed check reads `longestBridge` against the 3,072-token window (the Colab cell refuses bridged metrics without it).

In [ ]:
#| export
def bridgeStepStats(gs, tok=None):
    """A step's rollouts [(LLMConfig, Rollout)] (prompts kept) -> {longestBridge: a bridge's longest prompt plus reply in tokens (as
    pairLengths, eos aside; tok None: 4 characters a token), relaysPerBridgeTurn}; {} when no board has bridges."""
    from explRL.clue_policy import userText
    from explRL.local_policy import promptText
    enc = lambda x: len(tok(x, add_special_tokens=False)['input_ids'])
    n = (lambda s, bl, rep: enc(promptText(tok, s, bl)) + enc(rep)) if tok is not None else \
        (lambda s, bl, rep: (len(s) + 2 + len(userText(bl)) + len(rep)) // 4)
    longest, turns, relays, seen = 0, 0, 0, False
    for _, ro in gs:
        for b, bd in enumerate(ro.boards):
            if not getattr(bd, 'bridgeSet', None):
                continue
            seen, relays = True, relays + len(bd.relayLog)
            turns += sum(1 for row in bd.ledger if row[0] in bd.bridgeSet)
            for i in bd.bridgeSet:
                for (s, bl), x in zip(ro.prompts[b][i], ro.logs[b][i]):
                    longest = max(longest, n(s, bl, x.get('response') or ''))
    return dict(longestBridge=int(longest), relaysPerBridgeTurn=relays / turns if turns else NAN) if seen else {}

## Tests

In [ ]:
# test_bridge19_world (T5a, T5b; P19-1, P19-8, P19-9): bridge1 in two squads or more, a single squad unbridged; Gate C1's
# record sets the world; every arm's config builds a bridge board, with BridgeContestLog under contest credit
from explRL.bridge_board import BridgeBoardEnv, checkBridge
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_roll import boardFactory
from explRL.contest import ContestLog
from explRL.llm_runs import LLMConfig
from explRL.rl19_bridge import *
from explRL.rl19_grid import ARMS19, RL19_PLAN, rl19Config, worldFor19
from explRL.rl19_step import check19, world19
assert dict(BRIDGE19) == dict(couple=0.25, bridges=1) and RL19_PLAN['gates']['C1'] is GATE_C1
assert worldFor19(GATE_C1) == RL19_PLAN['world'] == 'bridges' and worldFor19(dict(GATE_C1, passed=False)) == 'squads'
assert worldFor19(dict(GATE_C1, inertFired=True)) == 'squads' and worldFor19(dict(GATE_C1, inertFired=True, userAcceptsBridges=True)) == 'bridges'
assert bridgeWorld19(dict(squads=4), 16) == dict(squads=4, couple=0.25, bridges=1) and bridgeWorld19(dict(squads=8), 8) == dict(squads=8)
assert bridgeWorld19((('r', 2), ('squads', 8)), 16) == (('r', 2), ('squads', 8), ('couple', 0.25), ('bridges', 1))
assert isBridge19(dict(scheme='contest', couple=0.25)) and isBridge19(dict(p19=True, couple=0.25))
assert not isBridge19(dict(scheme='individual', couple=0.25, bridges=1)) and not isBridge19(dict(scheme='contest'))   # C1's rows; unbridged


def lcfg(arm, S, nl, life=30):
    "A training board's LLMConfig as trainRL17 builds it (rl17_train's lcfg)."
    c = rl19Config(arm, 1, 'm', 'o', {}, S, plan=dict(RL19_PLAN, records=False))   # arm 4's records: P19-5, at the freeze
    check19(c)
    return c, LLMConfig(task='clue', channel=c.channel, N=16, nLiar=nl, turns=c.turns, model=c.model,
                        world={**{k: CLUE_SETTINGS[k] for k in BOARD}, 'scheme': c.scheme, **dict(c.world), **world19(c, life)})


for arm in ARMS19:
    for S in (4, 8):
        c, L = lcfg(arm, S, 6)
        w = dict(c.world)
        assert w['couple'] == 0.25 and w['bridges'] == 1 and 'bridgeGuard' not in w and checkBridge(L.world, 16, 6)
        bd = boardFactory(L)[0](900)
        assert isinstance(bd, BridgeBoardEnv) and len(bd.bridgeSet) == 16 // S
        assert type(bd.contestLog) is (BridgeContestLog if c.scheme == 'contest' else type(None))

In [ ]:
# test_bridge19_contest (FR-A9; P20-12, P23-8): on a hand-built 16-agent board in two squads (arm 4: contest credit open to
# all), a liar's mirror lie, relayed by the bridge, is the liar's on the receiving board (a plain ContestLog sees no lie); a
# true holder's mirror post is a border contest whose watchers are the bridge and the receiving squad's agent shown the
# relay; the bridge's relay of the true value contests the relayed lie; that agent's correct solve pays both contesters; a relayed true value splits its credit (P20-12)
import json
from explRL.contest import preventionBonus
from explRL.contest_eval import _columns, contestColumns, refuses
_, L = lcfg('a19open', 8, 4)
bd = boardFactory(L)[0](901)
log, nS = bd.contestLog, bd.nSquads


def pick(bd, skip=()):
    "A coupled open subtask with an honest holder of its cross clue, a liar in the mirror squad and a watcher in its own."
    for k in sorted(bd.coupled):
        s, s2 = bd.coupled[k]
        hs = [h for h in bd.tasks[k].holders[1] if h >= bd.nLiar]
        liars = [i for i in range(bd.nLiar) if i % nS == s2]
        ws = [a for a in range(bd.nLiar, 16) if a % nS == s and a not in bd.bridgeSet and a not in bd.tasks[k].holders[1]]
        if k in bd.open and k not in skip and hs and liars and ws:
            return k, hs[0], liars[0], bd.bridges[s][0], ws[0]


k, h, liar, br, a = pick(bd)
p, tr = bd.tasks[k].task.clue(1)
fr, T = (tr + 1) % p, iter(x / 100 for x in range(1, 100))   # turns 0.01 apart: no subtask expires meanwhile
step = lambda i, act: bd.apply(i, act, next(T))
step(liar, ('share', (k, 1, fr)))                         # the lie on the mirror
step(br, ('wait', None))                                  # the bridge is shown it on its pair's mirror
step(br, ('share', (k, 1, fr)))                           # ... and relays it
assert bd.relayLog and bd.relayLog[-1][5][1] == liar and log.lies(k, 1)[0][1:] == (fr, liar) and ContestLog.lies(log, k, 1) == []
step(a, ('wait', None))                                   # a, in the receiving squad, is shown the relayed lie
assert set(log.sawBorder[(k, 1)]) == {br, a} and set(log.sawLie[(k, 1)]) == {a}   # the bridge saw the lie on the mirror only
step(h, ('share', (k, 1, tr)))                            # the true holder's mirror post: a border contest
c = log.contests[log.border[0]]
assert len(log.border) == 1 == len(log.contests) and (c.who, c.liar, c.lie) == (h, liar, fr) and set(c.watchers) == {br, a}
step(h, ('share', (k, 1, tr)))                            # a repeat true mirror post is no new border contest (no farming)
assert len(log.border) == 1 == len(log.contests)
step(br, ('share', (k, 1, tr)))                           # the bridge relays the true value: a contest against the relayed lie
c2 = log.contests[-1]
assert len(log.contests) == 2 and (c2.who, c2.liar, c2.lie) == (br, liar, fr) and set(c2.watchers) == {a}
step(a, ('open', k))
x = bd.tasks[k].task.x
step(a, ('submit', x))                                    # a's correct solve pays both contesters
assert [(q[1], q[2], q[3]) for q in log.paid] == [(h, a, k), (br, a, k)]
assert all(abs(q[5] - preventionBonus(cc.t_c - cc.t_l, log.H)) < 1e-12 for q, cc in zip(log.paid, (c, c2)))
assert log.earned(16)[h] > 0 and log.earned(16)[br] > 0
# the columns: relay-aware on this cell, plain for C1's (no Phase 19 marker)
C, P = contestColumns(L, bd), _columns(L, bd)
assert C['liarPosts'] == 1 and P['liarPosts'] == 0 and C['contests'] == 1 and P['contests'] == 0
assert C['borderContests'] == 1 and C['mirrorLies'] == 1 and json.loads(C['agentBorderContests'])[h] == 1
assert json.loads(C['agentShares']) == json.loads(P['agentShares']) and json.loads(C['agentShares'])[br] == 2
assert json.loads(C['agentContests'])[br] == 1 and json.loads(C['agentContests'])[h] == 0   # the contester is the real poster, as paid
assert C['liarFirst'] == 1 and C['uncorrected'] == 0 and P['liarFirst'] == 0 and C['misreport19'] != C['misreport19']   # relays only: NaN
assert not refuses(bd, a, k, x, 3)                        # P19-9: a coupled answer is exempt from enforceK
# a lie posted straight onto the receiving board (no relay) makes a board watcher, not a border watcher
k3, h3, liar3, br3, a3 = pick(bd, skip={k})
s3 = bd.coupled[k3][0]
p3, tr3 = bd.tasks[k3].task.clue(1)
own = next(i for i in range(bd.nLiar) if i % nS == s3)
step(own, ('share', (k3, 1, (tr3 + 1) % p3)))             # a liar of the receiving squad, on its own board
step(liar3, ('share', (k3, 1, (tr3 + 1) % p3)))           # the same false value on the mirror
step(a3, ('wait', None))
assert a3 in log.sawLie[(k3, 1)] and a3 not in log.sawBorder.get((k3, 1), {})
step(h3, ('share', (k3, 1, tr3)))
assert a3 not in log.contests[log.border[-1]].watchers and len(log.border) == 2
from explRL.bridge_grid import bridgeCheckCells           # P20-11's check runs in A1's world (p19): relay-aware columns too
assert all(isBridge19(c['cfg']['world']) for c in bridgeCheckCells())
# P20-12: a relayed true value's credit share goes half to the bridge, half to the first true poster on the mirror
k2, h2, _, br2, a2 = pick(bd, skip={k, k3})
p2, tr2 = bd.tasks[k2].task.clue(1)
step(h2, ('share', (k2, 1, tr2)))
step(br2, ('share', (k2, 1, tr2)))
before = bd.pending.copy()
step(a2, ('open', k2))
step(a2, ('submit', bd.tasks[k2].task.x))
got = bd.pending - before
assert not any(s[2] == k2 and s[3] == 0 for s in bd.shareLog)   # clue 1 is the only credited clue
assert abs(got[br2] - got[h2]) < 1e-12 and abs(got[br2] - bd.credit * bd.tasks[k2].value / 2) < 1e-12

In [ ]:
# test_bridge19_step (T5a done-when): a recorded 16-agent bridged rollout (the fake Claude client, prompts kept) gives the
# step's longestBridge (at least the bridge's longest prompt, bridgePromptTokens) and relays per bridge turn; none unbridged
import math
from explRL.bridge_agents import bridgePromptTokens
from explRL.clue_policy import claudePolicy
from explRL.clue_roll import rollout
from explRL.colony_agents import scriptedFaults
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply
_, L = lcfg('a19contest', 4, 4)
pol = claudePolicy(LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply)))
ro = rollout(L, [902], pol, scriptedFaults(L.world, 4), keepPrompts=True)
st = bridgeStepStats([(L, ro)])
assert set(st) == {'longestBridge', 'relaysPerBridgeTurn'} and st['longestBridge'] >= bridgePromptTokens(L, [902])['bridge'] > 0
assert math.isfinite(st['relaysPerBridgeTurn']) and st['relaysPerBridgeTurn'] == len(ro.boards[0].relayLog) / sum(
    1 for r in ro.boards[0].ledger if r[0] in ro.boards[0].bridgeSet)
L0 = LLMConfig.fromDict({**L.asDict(), 'world': {k: v for k, v in L.world.items() if k not in ('couple', 'bridges')}})
assert bridgeStepStats([(L0, rollout(L0, [902], pol, scriptedFaults(L0.world, 4), keepPrompts=True))]) == {}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()